# 🎨 AWS Architecture Graph Visualization Lab

Este notebook te permite visualizar gráficamente cualquier diagrama de arquitectura del proyecto utilizando el motor de renderizado de `graph_renderer`.

El motor de renderizado lee un archivo `.graphml`, calcula una distribución jerárquica para los nodos, inyecta colores representativos por flujo, coloca los logotipos oficiales de los servicios de AWS y exporta una imagen PNG de alta resolución.

In [ ]:
import os
import sys
import subprocess
import shutil
from pathlib import Path
from IPython.display import Image, display

# Configurar la ruta raíz del proyecto
PROJECT_ROOT = Path('/home/stemjara/Projects/AWS-Architecture')
sys.path.append(str(PROJECT_ROOT))

LAB_DIR = PROJECT_ROOT / 'whiteboard_selection_lab'
RENDERER_JS = PROJECT_ROOT / 'graph_renderer' / 'render_graph.mjs'

print('✓ Entorno de visualización de grafos inicializado.')

### 1. Función de Renderizado General

La función de abajo toma la ruta de cualquier archivo `.graphml`, lo procesa usando la herramienta headless Cytoscape en Node.js y muestra la imagen PNG directamente en el notebook.

In [ ]:
def visualizar_grafo(grafo_path: str | Path):
    """
    Toma la ruta de un archivo .graphml, lo procesa con render_graph.mjs
    y visualiza la imagen PNG resultante en el notebook.
    """
    grafo_path = Path(grafo_path).resolve()
    if not grafo_path.exists():
        print(f'❌ Error: No se encontró el archivo de grafo en: {grafo_path}')
        return
        
    basename = grafo_path.stem
    # Limpiar caracteres especiales de la ruta para el basename del archivo de salida
    basename_clean = str(basename).replace('/', '_').replace('\\', '_')
    
    # El renderer de JS escribe las salidas en graph_renderer/graphs_output/
    output_png = PROJECT_ROOT / 'graph_renderer' / 'graphs_output' / f'{basename_clean}.png'
    
    # Limpiar el PNG anterior si existe para asegurar la actualización visual
    if output_png.exists():
        output_png.unlink()
        
    # Copiar temporalmente a la carpeta graphs_input del renderer para simplificar la ejecución de Node
    temp_input_path = PROJECT_ROOT / 'graph_renderer' / 'graphs_input' / f'{basename_clean}.graphml'
    shutil.copy2(grafo_path, temp_input_path)
    
    # Ejecutar Node.js utilizando la ruta absoluta del archivo copiado
    cmd = ['node', str(RENDERER_JS), str(temp_input_path)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    # Eliminar el archivo temporal copiado
    if temp_input_path.exists():
        temp_input_path.unlink()
        
    if result.returncode == 0 and output_png.exists():
        print(f'✅ Renderizado con éxito: {output_png.name}')
        display(Image(filename=str(output_png)))
    else:
        print(f'❌ Error al renderizar grafo: {grafo_path.name}')
        print('stdout:', result.stdout)
        print('stderr:', result.stderr)

### 2. Previsualización y Comparación de los 4 Modelos

Al ingresar un `VIDEO_ID` (por ejemplo, el video en español `CsD5bmM6mpY`), el script buscará los grafos de sus 4 variantes correspondientes y los mostrará de forma secuencial:
1. **Ground Truth:** El diagrama de arquitectura de referencia manual (FAST25).
2. **Standard Gemini:** El grafo generado por el pipeline clásico.
3. **Parsimonious:** El grafo generado bajo el principio de parsimonia.
4. **Nueva Prueba (Test):** El grafo experimental que generaste en el laboratorio local.

In [ ]:
# Elige el VIDEO_ID a visualizar
VIDEO_ID = 'CsD5bmM6mpY'

def previsualizar_cuatro_modelos(video_id: str):
    print(f'🔍 Iniciando previsualización de modelos para el video: {video_id}\n')
    
    # Definir las rutas de origen para cada uno de los 4 modelos
    modelos = {
        '1. Ground Truth (Cloudscape)': PROJECT_ROOT / 'data' / 'cloudscape_gt' / f'{video_id}.graphml',
        '2. Standard Gemini': PROJECT_ROOT / 'data' / 'graphs' / f'{video_id}.graphml',
        '3. Parsimonious': PROJECT_ROOT / 'data' / 'graphs_parsimonious' / f'{video_id}.graphml',
        '4. Nueva Prueba (Test - Lab Workspace)': LAB_DIR / 'lab_workspace' / video_id / 'test_graph.graphml'
    }
    
    modelos_encontrados = 0
    for titulo, ruta_origen in modelos.items():
        print(f'\n' + '=' * 65)
        print(f'  {titulo}')
        print('=' * 65)
        
        if ruta_origen.exists():
            # Asignamos un nombre específico agregando el sufijo del modelo al renderizar
            sufijo = titulo.split('.')[0].strip().replace(' ', '_').lower()
            nombre_render = f'{video_id}_modelo_{sufijo}.graphml'
            
            # Crear una copia con nombre único en una carpeta temporal del workspace local
            temp_workspace_dir = LAB_DIR / 'lab_workspace' / video_id / 'temp_visualize'
            temp_workspace_dir.mkdir(parents=True, exist_ok=True)
            
            ruta_temp = temp_workspace_dir / nombre_render
            shutil.copy2(ruta_origen, ruta_temp)
            
            # Ejecutar visualización
            visualizar_grafo(ruta_temp)
            modelos_encontrados += 1
            
            # Limpieza del archivo temporal local
            if ruta_temp.exists():
                ruta_temp.unlink()
        else:
            print(f'⚠️  El modelo \'{titulo}\' no está disponible (archivo no encontrado en {ruta_origen.name}).')
            
    print(f'\n🎉 Proceso completado. Se visualizaron {modelos_encontrados}/4 modelos.')

# Ejecutar la previsualización
previsualizar_cuatro_modelos(VIDEO_ID)